# 0. Install dependencies

In [2]:
!pip install lxml requests gender-guesser

# 1. Parse names

In [10]:
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
import json

# Define the namespace
namespaces = {
    'tei': 'http://www.tei-c.org/ns/1.0',
    'telota': 'http://www.telota.de',
    'xs': 'http://www.w3.org/2001/XMLSchema'
}

# Define the path to the Personen directory (relative to notebook location)
personen_dir = Path('../data/raw/schleiermacher/Register/Personen')

# Load all XML files and parse person entries
all_persons = []

for xml_file in personen_dir.glob('*.xml'):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Find all person elements
        persons = root.findall('.//tei:person', namespaces)

        for person in persons:
            person_data = {
                'xml_id': person.get('{http://www.w3.org/XML/1998/namespace}id')
            }

            # Extract GND URI
            idno = person.find('tei:idno[@type="uri"]', namespaces)
            if idno is not None:
                person_data['gnd_uri'] = idno.text

            # Extract regular name (surname and forename)
            persName_reg = person.find('tei:persName[@type="reg"]', namespaces)
            if persName_reg is not None:
                surname = persName_reg.find('tei:surname', namespaces)
                forename = persName_reg.find('tei:forename', namespaces)
                person_data['surname'] = surname.text if surname is not None else None
                person_data['forename'] = forename.text if forename is not None else None

            # Extract alternative names
            alt_names = []
            for persName_alt in person.findall('tei:persName[@type="alt"]', namespaces):
                alt_name = {}
                subtype = persName_alt.get('subtype')
                if subtype:
                    alt_name['subtype'] = subtype

                # Check for surname/forename structure
                surname_alt = persName_alt.find('tei:surname', namespaces)
                forename_alt = persName_alt.find('tei:forename', namespaces)
                name_alt = persName_alt.find('tei:name', namespaces)

                if surname_alt is not None:
                    alt_name['surname'] = surname_alt.text
                if forename_alt is not None:
                    alt_name['forename'] = forename_alt.text
                if name_alt is not None:
                    alt_name['name'] = name_alt.text

                alt_names.append(alt_name)

            if alt_names:
                person_data['alt_names'] = alt_names

            # Extract birth and death years
            birth = person.find('tei:birth', namespaces)
            death = person.find('tei:death', namespaces)
            person_data['birth'] = birth.text if birth is not None and birth.text else None
            person_data['death'] = death.text if death is not None and death.text else None

            # Extract note
            note = person.find('tei:note', namespaces)

            # replace newlines and tabs in note text with spaces
            if note is not None and note.text:
                cleaned_note = note.text.replace('\n', ' ').replace('\t', ' ')

                # replace multiple spaces with a single space
                cleaned_note = ' '.join(cleaned_note.split())

                person_data['note'] = cleaned_note
            else:
                person_data['note'] = None

            all_persons.append(person_data)

        print(f"Loaded {len(persons)} persons from: {xml_file.name}")

    except ET.ParseError as e:
        print(f"Error parsing {xml_file.name}: {e}")

print(f"\nTotal persons loaded: {len(all_persons)}")

# Create DataFrame
df_persons = pd.DataFrame(all_persons)

print(f"\nDataFrame shape: {df_persons.shape}")
print(f"\nFirst few rows:")
print(df_persons.head())

# Save to CSV
output_csv = Path('../data/raw/schleiermacher/persons.csv')

# Ensure the output directory exists
output_csv.parent.mkdir(parents=True, exist_ok=True)


df_persons['alt_names'] = df_persons['alt_names'].apply(
    lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
)
df_persons.to_csv(output_csv, index=False)
print(f"\nData saved to: {output_csv}")

Loaded 49 persons from: U.xml
Loaded 469 persons from: W.xml
Loaded 1214 persons from: S.xml
Loaded 391 persons from: L.xml
Loaded 183 persons from: E.xml
Loaded 126 persons from: V.xml
Loaded 434 persons from: P.xml
Loaded 631 persons from: H.xml
Loaded 431 persons from: G.xml
Loaded 516 persons from: K.xml
Loaded 7 persons from: Q.xml
Loaded 159 persons from: N.xml
Loaded 239 persons from: C.xml
Loaded 98 persons from: Z.xml
Loaded 496 persons from: R.xml
Loaded 2 persons from: Y.xml
Loaded 4 persons from: X.xml
Loaded 305 persons from: A.xml
Loaded 47 persons from: I.xml
Loaded 518 persons from: M.xml
Loaded 256 persons from: T.xml
Loaded 816 persons from: B.xml
Loaded 325 persons from: D.xml
Loaded 125 persons from: J.xml
Loaded 101 persons from: O.xml
Loaded 331 persons from: F.xml

Total persons loaded: 8273

DataFrame shape: (8273, 8)

First few rows:
     xml_id                           gnd_uri     surname  \
0  S1121639  https://d-nb.info/gnd/1047176823      Uebele   
1  S412